In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# -*- coding: utf-8 -*-
"""
Multitask ClinicalBERT:
- Mortality: binary classification (hospital_expire_flag)
- LOS: MULTICLASS classification via los_days bins

Outputs:
- Mortality: F1 (%), AUROC (%) + 95% bootstrap CI
- LOS multiclass: Macro F1 (%), AUROC (%) (OvR macro) + 95% bootstrap CI

Added:
- Subgroup (race) performance on TEST:
  - Per-race metrics table for mortality + LOS
  - Disparity measures (max-min gap across groups) + bootstrap CI
- Calibration curves by race:
  - Mortality: reliability diagram, Brier score, ECE by race
  - LOS: top-label confidence calibration (accuracy vs confidence), multiclass Brier, top-label ECE by race

Notes:
- AUROC/AP are undefined if y_true has only one class (e.g., no positives in a subgroup).
  We return np.nan to avoid warnings and to keep bootstrap robust.
"""

import os, inspect, ast, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
    set_seed, EvalPrediction,
    EarlyStoppingCallback
)

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, brier_score_loss
)
from sklearn.preprocessing import label_binarize
from sklearn.exceptions import UndefinedMetricWarning

# Headless plotting (Kaggle/servers)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Optional: silence sklearn warnings (the safe_* wrappers prevent these anyway)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", message="No positive class found in y_true*")


# ------------------------------
# Config
# ------------------------------
@dataclass
class CFG:
    csv_path: str = "/content/drive/MyDrive/EHR/dx_encoded_feats.csv"
    model_name: str = "emilyalsentzer/Bio_ClinicalBERT"
    seed: int = 42

    # tokenization
    max_len: int = 512
    max_dx: int = 60

    # split
    test_size: float = 0.15
    val_size: float = 0.15

    # task weights
    w_mort: float = 1.0
    w_los: float = 1.0

    # LOS bins
    los_bin_edges: tuple = (0.0, 3.0, 7.0, 14.0, float("inf"))
    los_bin_labels: tuple = ("0-3", "3-7", "7-14", "14+")

    # training
    epochs: int = 20
    lr: float = 2e-5
    train_bs: int = 8
    eval_bs: int = 16
    grad_accum: int = 2
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    lr_scheduler_type: str = "cosine"
    max_grad_norm: float = 1.0
    early_stop_patience: int = 8

    # mortality imbalance
    use_focal: bool = True
    focal_gamma: float = 2.0
    focal_alpha: float = 0.25

    # structured features
    use_structured_feats: bool = True  # age + n_dx

    # optional tokenizer expansion
    add_code_tokens_to_tokenizer: bool = False
    max_added_tokens: int = 3000

    # mortality threshold policy
    threshold_mode: str = "f1"  # "f1" | "recall" | "precision"
    target_recall: float = 0.70
    target_precision: float = 0.50

    # bootstrap CI
    n_boot: int = 2000
    ci_alpha: float = 0.05

    # subgroup evaluation
    min_group_n: int = 50  # minimum subgroup size to compute subgroup bootstrap CIs

    # calibration
    calib_bins: int = 10
    calib_topk_overlay: int = 6

    # save
    save_dir: str = "llm_multitask_saved"


cfg = CFG()
set_seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Mixed precision choice
use_bf16, use_fp16 = False, False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    use_bf16 = (major >= 8)      # A100+
    use_fp16 = not use_bf16      # T4/V100
print("bf16:", use_bf16, "fp16:", use_fp16)


# ------------------------------
# Safe metric wrappers (fix AUROC/AP warnings)
# ------------------------------
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if np.unique(y_true).size < 2:
        return np.nan
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

def safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    # AP undefined / not meaningful if no positives or only one class
    if np.unique(y_true).size < 2 or y_true.sum() == 0:
        return np.nan
    try:
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan


# ------------------------------
# Version-proof TrainingArguments builder
# ------------------------------
def make_training_args(**kwargs):
    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters
    if "eval_strategy" in params and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    if "evaluation_strategy" in params and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    filtered = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**filtered)


# ------------------------------
# Bootstrap CI utilities
# ------------------------------
def bootstrap_ci(y_true, x, metric_fn, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    x = np.asarray(x)
    n = len(y_true)

    # Point estimate
    try:
        point = metric_fn(y_true, x)
    except Exception:
        point = np.nan

    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        xs = x[idx]
        try:
            v = metric_fn(yt, xs)
            if np.isfinite(v):
                stats.append(v)
        except Exception:
            pass

    stats = np.asarray(stats)
    if len(stats) == 0:
        return float(point) if np.isfinite(point) else np.nan, np.nan, np.nan
    lo = np.quantile(stats, alpha / 2)
    hi = np.quantile(stats, 1 - alpha / 2)
    return float(point), float(lo), float(hi)

def pct3(t):
    return (100.0 * t[0], 100.0 * t[1], 100.0 * t[2])

def fmt_ci(p, lo, hi, d=2):
    return f"{p:.{d}f} ({lo:.{d}f}, {hi:.{d}f})"


# ------------------------------
# Subgroup (race) performance utilities
# ------------------------------
def _safe_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn, fp, fn, tp

def tpr_fpr(y_true, y_pred):
    tn, fp, fn, tp = _safe_confusion(y_true, y_pred)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return float(tpr), float(fpr)

def bootstrap_gap_by_group(y_true, x, groups, metric_fn,
                           n_boot=2000, alpha=0.05, seed=42, min_n=20):
    """
    Disparity measure: gap = max(metric_g) - min(metric_g), with bootstrap CI.
    Bootstraps within each group to keep group sizes fixed.
    Groups with < min_n are ignored.
    Undefined subgroup metric values are ignored.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    x = np.asarray(x)
    groups = np.asarray(groups)

    grp_idx = {g: np.where(groups == g)[0] for g in np.unique(groups)}
    grp_idx = {g: idx for g, idx in grp_idx.items() if len(idx) >= min_n}

    def gap_stat(y, x, groups):
        vals = []
        for g, idx in grp_idx.items():
            try:
                v = metric_fn(y[idx], x[idx])
                if np.isfinite(v):
                    vals.append(v)
            except Exception:
                pass
        if len(vals) < 2:
            return np.nan
        return float(np.max(vals) - np.min(vals))

    point = gap_stat(y_true, x, groups)

    stats = []
    for _ in range(n_boot):
        if not grp_idx:
            break

        boot_parts = [rng.choice(idx, size=len(idx), replace=True) for idx in grp_idx.values()]
        boot_idx = np.concatenate(boot_parts)

        yt = y_true[boot_idx]
        xt = x[boot_idx]
        gt = groups[boot_idx]

        vals = []
        for g in np.unique(gt):
            gi = np.where(gt == g)[0]
            if len(gi) < min_n:
                continue
            try:
                v = metric_fn(yt[gi], xt[gi])
                if np.isfinite(v):
                    vals.append(v)
            except Exception:
                pass

        if len(vals) >= 2:
            stats.append(np.max(vals) - np.min(vals))

    stats = np.asarray(stats)
    if len(stats) == 0:
        return float(point) if np.isfinite(point) else np.nan, np.nan, np.nan
    lo = np.quantile(stats, alpha / 2)
    hi = np.quantile(stats, 1 - alpha / 2)
    return float(point), float(lo), float(hi)

def subgroup_metrics_table(
    df_test,
    mort_prob, mort_pred,
    los_prob, los_pred,
    n_boot=2000, alpha=0.05, seed=42, min_n_ci=50
):
    """
    Per-race performance table (+ subgroup bootstrap CIs where possible).
    Uses safe_roc_auc/safe_ap to avoid undefined-metric warnings.
    """
    races = df_test["race"].fillna("UNKNOWN").astype(str).values
    rows = []

    def ci_if_ok(y, x, fn, n):
        if n < min_n_ci:
            return (np.nan, np.nan)
        return bootstrap_ci(y, x, fn, n_boot=n_boot, alpha=alpha, seed=seed)[1:]

    for race in sorted(pd.unique(races)):
        idx = np.where(races == race)[0]
        n = len(idx)
        if n == 0:
            continue

        y_m = df_test.iloc[idx]["y_mort"].values.astype(int)
        y_l = df_test.iloc[idx]["y_los_cls"].values.astype(int)

        mp = mort_prob[idx]
        mhat = mort_pred[idx]
        lp = los_prob[idx]
        lhat = los_pred[idx]

        # Mortality
        mort_rate = float(np.mean(y_m))
        m_f1 = f1_score(y_m, mhat, zero_division=0)
        m_prec = precision_score(y_m, mhat, zero_division=0)
        m_rec = recall_score(y_m, mhat, zero_division=0)
        m_tpr, m_fpr = tpr_fpr(y_m, mhat)

        m_auc = safe_roc_auc(y_m, mp)
        m_ap = safe_ap(y_m, mp)

        m_f1_lo, m_f1_hi = ci_if_ok(y_m, mhat, lambda yt, yp: f1_score(yt, yp, zero_division=0), n)
        m_auc_lo, m_auc_hi = ci_if_ok(y_m, mp, lambda yt, ps: safe_roc_auc(yt, ps), n)
        m_ap_lo, m_ap_hi = ci_if_ok(y_m, mp, lambda yt, ps: safe_ap(yt, ps), n)

        # LOS
        los_macro_f1 = f1_score(y_l, lhat, average="macro", zero_division=0)
        try:
            los_auc = multiclass_auc_ovr_macro(y_l, lp)
        except Exception:
            los_auc = np.nan

        los_f1_lo, los_f1_hi = ci_if_ok(
            y_l, lhat,
            lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0), n
        )
        try:
            los_auc_lo, los_auc_hi = ci_if_ok(
                y_l, lp,
                lambda yt, prob: multiclass_auc_ovr_macro(yt, prob), n
            )
        except Exception:
            los_auc_lo, los_auc_hi = np.nan, np.nan

        rows.append({
            "race": race, "n": n, "mort_rate": mort_rate,
            "mort_auc": m_auc, "mort_auc_lo": m_auc_lo, "mort_auc_hi": m_auc_hi,
            "mort_ap": m_ap, "mort_ap_lo": m_ap_lo, "mort_ap_hi": m_ap_hi,
            "mort_f1": m_f1, "mort_f1_lo": m_f1_lo, "mort_f1_hi": m_f1_hi,
            "mort_precision": m_prec, "mort_recall": m_rec,
            "mort_tpr": m_tpr, "mort_fpr": m_fpr,
            "los_macro_f1": los_macro_f1, "los_f1_lo": los_f1_lo, "los_f1_hi": los_f1_hi,
            "los_auc": los_auc, "los_auc_lo": los_auc_lo, "los_auc_hi": los_auc_hi
        })

    out = pd.DataFrame(rows)
    return out.sort_values("n", ascending=False).reset_index(drop=True)


# ------------------------------
# Calibration utilities (by race)
# ------------------------------
def calibration_bins(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.digitize(y_prob, edges[1:-1], right=False)

    mean_pred = np.full(n_bins, np.nan, dtype=float)
    frac_pos = np.full(n_bins, np.nan, dtype=float)
    counts = np.zeros(n_bins, dtype=int)

    for b in range(n_bins):
        m = (bin_id == b)
        counts[b] = int(m.sum())
        if counts[b] > 0:
            mean_pred[b] = float(np.mean(y_prob[m]))
            frac_pos[b] = float(np.mean(y_true[m]))
    return edges, mean_pred, frac_pos, counts

def ece_from_bins(mean_pred, frac_pos, counts):
    mean_pred = np.asarray(mean_pred)
    frac_pos = np.asarray(frac_pos)
    counts = np.asarray(counts)
    n = counts.sum()
    if n == 0:
        return np.nan
    mask = counts > 0
    return float(np.sum(np.abs(frac_pos[mask] - mean_pred[mask]) * (counts[mask] / n)))

def multiclass_toplabel_calibration(y_true_cls, prob_2d, n_bins=10):
    y_true_cls = np.asarray(y_true_cls).astype(int)
    prob_2d = np.asarray(prob_2d).astype(float)

    conf = prob_2d.max(axis=1)
    pred = prob_2d.argmax(axis=1)
    correct = (pred == y_true_cls).astype(int)

    edges, mean_conf, acc, counts = calibration_bins(correct, conf, n_bins=n_bins)
    return edges, mean_conf, acc, counts

def multiclass_brier(y_true_cls, prob_2d):
    y_true_cls = np.asarray(y_true_cls).astype(int)
    prob_2d = np.asarray(prob_2d).astype(float)
    n, K = prob_2d.shape
    Y = np.zeros((n, K), dtype=float)
    Y[np.arange(n), y_true_cls] = 1.0
    return float(np.mean(np.sum((prob_2d - Y) ** 2, axis=1)))

def plot_reliability(ax, mean_pred, frac_pos, counts, label=None):
    mask = counts > 0
    x = mean_pred[mask]
    y = frac_pos[mask]
    denom = max(counts[mask].sum(), 1)
    s = 25 + 220 * (counts[mask] / denom)
    ax.scatter(x, y, s=s, alpha=0.85, label=label)

def calibration_by_race_binary(
    y_true, y_prob, race,
    out_dir,
    n_bins=10,
    min_n=50,
    top_k_overlay=6,
    prefix="mortality"
):
    os.makedirs(out_dir, exist_ok=True)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    race = pd.Series(race).fillna("UNKNOWN").astype(str).values

    rows = []
    for r in sorted(pd.unique(race)):
        idx = np.where(race == r)[0]
        n = len(idx)
        if n == 0:
            continue
        yt = y_true[idx]
        yp = y_prob[idx]

        # Brier score: defined even if single class, but sklearn may not like edge cases; fallback safe.
        try:
            bs = brier_score_loss(yt, yp) if len(np.unique(yt)) > 1 else float(np.mean((yp - yt) ** 2))
        except Exception:
            bs = float(np.mean((yp - yt) ** 2))

        _, mp, fp, c = calibration_bins(yt, yp, n_bins=n_bins)
        ece = ece_from_bins(mp, fp, c)

        rows.append({"race": r, "n": n, "brier": float(bs), "ece": float(ece)})

    metrics = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
    metrics_path = os.path.join(out_dir, f"{prefix}_calibration_metrics_by_race.csv")
    metrics.to_csv(metrics_path, index=False)

    print(f"\n[{prefix}] Calibration metrics by race (saved to {metrics_path}):")
    print(metrics.to_string(index=False))

    # Overlay for top-k
    top = metrics.head(top_k_overlay)["race"].tolist()
    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
    ax.set_title(f"{prefix.capitalize()} calibration (top {top_k_overlay} races)")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed event rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    for r in top:
        idx = np.where(race == r)[0]
        yt = y_true[idx]
        yp = y_prob[idx]
        _, mp, fp, c = calibration_bins(yt, yp, n_bins=n_bins)
        plot_reliability(ax, mp, fp, c, label=f"{r} (n={len(idx)})")

    ax.legend(fontsize=8, loc="best")
    overlay_path = os.path.join(out_dir, f"{prefix}_calibration_overlay_top{top_k_overlay}.png")
    fig.tight_layout()
    fig.savefig(overlay_path, dpi=200)
    plt.close(fig)

    # Per-race plots (only if enough data)
    for r in metrics.loc[metrics["n"] >= min_n, "race"].tolist():
        idx = np.where(race == r)[0]
        yt = y_true[idx]
        yp = y_prob[idx]
        _, mp, fp, c = calibration_bins(yt, yp, n_bins=n_bins)

        fig, ax = plt.subplots(figsize=(6.0, 5.5))
        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        plot_reliability(ax, mp, fp, c, label=f"{r} (n={len(idx)})")
        ax.set_title(f"{prefix.capitalize()} calibration: {r}")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed event rate")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=9, loc="best")
        fig.tight_layout()
        safe_r = r.replace(" ", "_").replace("/", "_")
        out_path = os.path.join(out_dir, f"{prefix}_calibration_{safe_r}.png")
        fig.savefig(out_path, dpi=200)
        plt.close(fig)

    return metrics

def calibration_by_race_multiclass(
    y_true_cls, prob_2d, race,
    out_dir,
    n_bins=10,
    min_n=50,
    top_k_overlay=6,
    prefix="los"
):
    os.makedirs(out_dir, exist_ok=True)
    y_true_cls = np.asarray(y_true_cls).astype(int)
    prob_2d = np.asarray(prob_2d).astype(float)
    race = pd.Series(race).fillna("UNKNOWN").astype(str).values

    rows = []
    for r in sorted(pd.unique(race)):
        idx = np.where(race == r)[0]
        n = len(idx)
        if n == 0:
            continue
        yt = y_true_cls[idx]
        pp = prob_2d[idx]

        bs = multiclass_brier(yt, pp)
        _, mc, acc, c = multiclass_toplabel_calibration(yt, pp, n_bins=n_bins)
        ece = ece_from_bins(mc, acc, c)

        rows.append({"race": r, "n": n, "brier_mc": float(bs), "ece_toplabel": float(ece)})

    metrics = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
    metrics_path = os.path.join(out_dir, f"{prefix}_calibration_metrics_by_race.csv")
    metrics.to_csv(metrics_path, index=False)

    print(f"\n[{prefix}] Calibration metrics by race (saved to {metrics_path}):")
    print(metrics.to_string(index=False))

    # Overlay for top-k
    top = metrics.head(top_k_overlay)["race"].tolist()
    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
    ax.set_title(f"{prefix.upper()} calibration (top-label; top {top_k_overlay} races)")
    ax.set_xlabel("Mean confidence (max class probability)")
    ax.set_ylabel("Accuracy within bin")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    for r in top:
        idx = np.where(race == r)[0]
        yt = y_true_cls[idx]
        pp = prob_2d[idx]
        _, mc, acc, c = multiclass_toplabel_calibration(yt, pp, n_bins=n_bins)
        plot_reliability(ax, mc, acc, c, label=f"{r} (n={len(idx)})")

    ax.legend(fontsize=8, loc="best")
    overlay_path = os.path.join(out_dir, f"{prefix}_calibration_overlay_top{top_k_overlay}.png")
    fig.tight_layout()
    fig.savefig(overlay_path, dpi=200)
    plt.close(fig)

    # Per-race plots (only if enough data)
    for r in metrics.loc[metrics["n"] >= min_n, "race"].tolist():
        idx = np.where(race == r)[0]
        yt = y_true_cls[idx]
        pp = prob_2d[idx]

        _, mc, acc, c = multiclass_toplabel_calibration(yt, pp, n_bins=n_bins)

        fig, ax = plt.subplots(figsize=(6.0, 5.5))
        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        plot_reliability(ax, mc, acc, c, label=f"{r} (n={len(idx)})")
        ax.set_title(f"{prefix.upper()} calibration (top-label): {r}")
        ax.set_xlabel("Mean confidence (max class probability)")
        ax.set_ylabel("Accuracy within bin")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=9, loc="best")
        fig.tight_layout()
        safe_r = r.replace(" ", "_").replace("/", "_")
        out_path = os.path.join(out_dir, f"{prefix}_calibration_{safe_r}.png")
        fig.savefig(out_path, dpi=200)
        plt.close(fig)

    return metrics


# ------------------------------
# Dataset-specific helpers
# ------------------------------
def safe_get(row, col, default="NA"):
    v = row.get(col, default)
    if pd.isna(v):
        return default
    return v

def parse_list_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return []
    try:
        out = ast.literal_eval(s)
        return out if isinstance(out, list) else []
    except Exception:
        return []

def los_to_class(los_days, edges):
    for k in range(len(edges) - 1):
        if edges[k] <= los_days < edges[k + 1]:
            return k
    return len(edges) - 2

def build_admission_text(g: pd.DataFrame, max_dx: int = 60) -> str:
    g = g.sort_values("seq_num")
    r = g.iloc[0]
    header = (
        f"[AGE] {safe_get(r,'age_at_admit')} "
        f"[SEX] {safe_get(r,'gender')} "
        f"[RACE] {safe_get(r,'race')} "
        f"[INS] {safe_get(r,'insurance')} "
        f"[ATYPE] {safe_get(r,'admission_type')} "
        f"[ALOC] {safe_get(r,'admission_location')} "
    )

    toks = []
    for _, row in g.head(max_dx).iterrows():
        dx = str(row.get("dx_norm", "")).strip()
        if dx and dx.lower() != "nan" and dx != "NoDx":
            toks.append(f"DX_{dx.replace('.', '_')}")
        ccsr_codes = parse_list_str(row.get("ccsr_list_sorted", ""))
        for c in ccsr_codes[:2]:
            c = str(c).strip()
            if c:
                toks.append(f"CCSR_{c}")

    if not toks:
        for _, row in g.head(max_dx).iterrows():
            icd10 = str(row.get("icd10_full", "")).strip()
            if icd10 and icd10.lower() != "nan":
                toks.append(f"ICD10_{icd10.replace('.', '_')}")

    return header + " [DXSEQ] " + " ".join(toks)

def make_admission_level_df(df: pd.DataFrame, max_dx: int, edges) -> pd.DataFrame:
    rows = []
    for (subject_id, hadm_id), g in df.groupby(["subject_id", "hadm_id"], sort=False):
        g0 = g.iloc[0]
        y_mort = int(g0["hospital_expire_flag"])
        y_los = float(g0["los_days"])
        y_los_cls = int(los_to_class(y_los, edges))

        age_raw = g0.get("age_at_admit", np.nan)
        try:
            age = float(age_raw)
        except Exception:
            age = np.nan
        if np.isnan(age):
            age = 0.0
        n_dx = int(len(g))

        rows.append({
            "subject_id": int(subject_id),
            "hadm_id": int(hadm_id),
            "text": build_admission_text(g, max_dx=max_dx),
            "y_mort": y_mort,
            "y_los": y_los,
            "y_los_cls": y_los_cls,
            "race": str(g0.get("race", "UNKNOWN")),
            "age": age,
            "n_dx": n_dx
        })
    return pd.DataFrame(rows)


# ------------------------------
# Subject-wise stratified split by race
# ------------------------------
def split_subject_wise_stratified_by_race(df_adm: pd.DataFrame, seed: int, test_size: float, val_size: float):
    subj = (
        df_adm.groupby("subject_id")["race"]
        .agg(lambda x: x.value_counts().index[0])
        .reset_index()
        .rename(columns={"race": "race_subj"})
    )

    subjects = subj["subject_id"].values
    y_race = subj["race_subj"].values

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(sss1.split(subjects, y_race))
    trainval_subjects = set(subjects[trainval_idx])
    test_subjects = set(subjects[test_idx])

    df_test = df_adm[df_adm["subject_id"].isin(test_subjects)].reset_index(drop=True)
    df_trval = df_adm[df_adm["subject_id"].isin(trainval_subjects)].reset_index(drop=True)

    subj_trval = subj[subj["subject_id"].isin(trainval_subjects)].reset_index(drop=True)
    subjects_trval = subj_trval["subject_id"].values
    y_race_trval = subj_trval["race_subj"].values

    val_size_inside = val_size / (1.0 - test_size)
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_inside, random_state=seed + 1)
    tr_idx, val_idx = next(sss2.split(subjects_trval, y_race_trval))
    train_subjects = set(subjects_trval[tr_idx])
    val_subjects = set(subjects_trval[val_idx])

    df_train = df_adm[df_adm["subject_id"].isin(train_subjects)].reset_index(drop=True)
    df_val = df_adm[df_adm["subject_id"].isin(val_subjects)].reset_index(drop=True)

    return df_train, df_val, df_test


# ------------------------------
# Tokenizer + datasets
# ------------------------------
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else tokenizer.unk_token

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=cfg.max_len)

class DataCollatorMultiTask:
    def __init__(self, tokenizer, use_structured_feats=True):
        self.tokenizer = tokenizer
        self.use_structured_feats = use_structured_feats

    def __call__(self, features):
        y_mort = torch.tensor([f["y_mort"] for f in features], dtype=torch.float32)
        y_los_cls = torch.tensor([f["y_los_cls"] for f in features], dtype=torch.long)

        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["y_mort"] = y_mort
        batch["y_los_cls"] = y_los_cls

        if self.use_structured_feats:
            feats = torch.tensor(
                [[float(f.get("age", 0.0)), float(f.get("n_dx", 0.0))] for f in features],
                dtype=torch.float32
            )
            batch["feats"] = feats

        return batch


# ------------------------------
# Loss (mortality focal optional)
# ------------------------------
def focal_loss_with_logits(logits, targets, alpha=0.25, gamma=2.0, reduction="mean", pos_weight=None):
    device_ = logits.device
    targets = targets.to(device=device_, dtype=logits.dtype)
    if pos_weight is not None:
        pos_weight = pos_weight.to(device=device_, dtype=logits.dtype)

    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    p = torch.sigmoid(logits)
    pt = torch.where(targets == 1, p, 1 - p)
    a_t = torch.where(targets == 1, alpha, 1 - alpha)
    loss = a_t * (1 - pt) ** gamma * bce
    return loss.mean() if reduction == "mean" else loss.sum()


# ------------------------------
# Model
# ------------------------------
class AttnPool(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, x, mask):
        scores = self.score(x).squeeze(-1).float()
        scores = scores.masked_fill(mask == 0, -1e9)
        w = torch.softmax(scores, dim=1).to(dtype=x.dtype).unsqueeze(-1)
        return (x * w).sum(dim=1)

class MultiTaskLLM(nn.Module):
    def __init__(self, backbone_name: str, num_los_classes: int,
                 dropout: float = 0.1, pos_weight=None,
                 use_focal=False, focal_alpha=0.25, focal_gamma=2.0,
                 use_structured_feats=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        h = self.backbone.config.hidden_size
        self.pool = AttnPool(h)
        self.drop = nn.Dropout(dropout)
        self.use_structured_feats = use_structured_feats

        if self.use_structured_feats:
            self.feat_mlp = nn.Sequential(
                nn.Linear(2, 16),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            out_in = h + 16
        else:
            out_in = h

        self.mort_head = nn.Linear(out_in, 1)
        self.los_cls_head = nn.Linear(out_in, num_los_classes)

        if pos_weight is None:
            self.pos_weight = None
        else:
            self.register_buffer("pos_weight", pos_weight)

        self.use_focal = use_focal
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma

    def forward(self, input_ids, attention_mask, feats=None,
                y_mort=None, y_los_cls=None, w_mort=1.0, w_los=1.0,
                **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled = self.drop(self.pool(out.last_hidden_state, attention_mask))

        if self.use_structured_feats:
            if feats is None:
                feats = torch.zeros((pooled.size(0), 2), device=pooled.device, dtype=pooled.dtype)
            fused = torch.cat([pooled, self.feat_mlp(feats)], dim=1)
        else:
            fused = pooled

        mort_logits = self.mort_head(fused).squeeze(-1)
        los_logits = self.los_cls_head(fused)

        loss = None
        if (y_mort is not None) or (y_los_cls is not None):
            loss = 0.0
            if y_mort is not None:
                if self.use_focal:
                    mort_loss = focal_loss_with_logits(
                        mort_logits, y_mort.float(),
                        alpha=self.focal_alpha, gamma=self.focal_gamma,
                        pos_weight=self.pos_weight
                    )
                else:
                    mort_loss = F.binary_cross_entropy_with_logits(
                        mort_logits, y_mort.float(),
                        pos_weight=self.pos_weight
                    )
                loss = loss + w_mort * mort_loss

            if y_los_cls is not None:
                loss = loss + w_los * F.cross_entropy(los_logits, y_los_cls)

        return {"loss": loss, "mort_logits": mort_logits, "los_logits": los_logits}


# ------------------------------
# Custom Trainer
# ------------------------------
class MultiTaskTrainer(Trainer):
    def __init__(self, *args, w_mort=1.0, w_los=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.w_mort = w_mort
        self.w_los = w_los

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        out = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids", None),
            feats=inputs.get("feats", None),
            y_mort=inputs["y_mort"],
            y_los_cls=inputs["y_los_cls"],
            w_mort=self.w_mort,
            w_los=self.w_los
        )
        loss = out["loss"]
        return (loss, out) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                token_type_ids=inputs.get("token_type_ids", None),
                feats=inputs.get("feats", None),
                y_mort=inputs["y_mort"],
                y_los_cls=inputs["y_los_cls"],
                w_mort=self.w_mort,
                w_los=self.w_los
            )
        loss = outputs["loss"].detach()
        if prediction_loss_only:
            return (loss, None, None)
        preds = (outputs["mort_logits"].detach(), outputs["los_logits"].detach())
        labels = (inputs["y_mort"].detach(), inputs["y_los_cls"].detach())
        return (loss, preds, labels)


# ------------------------------
# Multiclass AUROC helper (OvR macro)
# ------------------------------
def multiclass_auc_ovr_macro(y_true_cls, prob_2d):
    y_true_cls = np.asarray(y_true_cls).astype(int)
    prob_2d = np.asarray(prob_2d)
    K = prob_2d.shape[1]
    Y = label_binarize(y_true_cls, classes=list(range(K)))
    return roc_auc_score(Y, prob_2d, average="macro", multi_class="ovr")


# ------------------------------
# compute_metrics for Trainer
# ------------------------------
def compute_metrics(p: EvalPrediction):
    mort_logits = p.predictions[0]
    los_logits = p.predictions[1]
    y_mort = p.label_ids[0]
    y_los_cls = p.label_ids[1]

    mort_prob = 1.0 / (1.0 + np.exp(-mort_logits))
    mort_auc = safe_roc_auc(y_mort, mort_prob)
    mort_ap = safe_ap(y_mort, mort_prob)
    mort_pred = (mort_prob >= 0.5).astype(int)
    mort_f1 = f1_score(y_mort, mort_pred, zero_division=0)

    expz = np.exp(los_logits - los_logits.max(axis=1, keepdims=True))
    los_prob = expz / expz.sum(axis=1, keepdims=True)
    los_pred = np.argmax(los_prob, axis=1)

    los_macro_f1 = f1_score(y_los_cls, los_pred, average="macro", zero_division=0)
    try:
        los_auc = multiclass_auc_ovr_macro(y_los_cls, los_prob)
    except Exception:
        los_auc = np.nan

    return {
        "mort_auc": mort_auc,
        "mort_ap": mort_ap,  # for model selection
        "mort_f1@0.5": mort_f1,
        "los_macro_f1": los_macro_f1,
        "los_auc": los_auc
    }


# ------------------------------
# Mortality threshold tuning
# ------------------------------
def find_best_threshold_by_f1(y_true, prob, t_min=0.001, t_max=0.99, steps=1200):
    ts = np.linspace(t_min, t_max, steps)
    f1s = [f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in ts]
    i = int(np.argmax(f1s))
    return float(ts[i]), float(f1s[i])

def threshold_for_target_recall(y_true, prob, target_recall=0.70, grid=2000):
    ts = np.linspace(0.999, 0.001, grid)
    for t in ts:
        pred = (prob >= t).astype(int)
        r = recall_score(y_true, pred, zero_division=0)
        if r >= target_recall:
            p = precision_score(y_true, pred, zero_division=0)
            f1v = f1_score(y_true, pred, zero_division=0)
            return float(t), float(p), float(r), float(f1v)
    return None

def threshold_for_target_precision(y_true, prob, target_precision=0.50, grid=2000):
    ts = np.linspace(0.001, 0.999, grid)
    for t in ts:
        pred = (prob >= t).astype(int)
        p = precision_score(y_true, pred, zero_division=0)
        if p >= target_precision:
            r = recall_score(y_true, pred, zero_division=0)
            f1v = f1_score(y_true, pred, zero_division=0)
            return float(t), float(p), float(r), float(f1v)
    return None


@torch.no_grad()
def predict_mort_probs(model, tokenizer, df_part, max_len, batch_size=32, use_structured_feats=True):
    model.eval()
    logits_all = []
    for i in range(0, len(df_part), batch_size):
        bdf = df_part.iloc[i:i + batch_size]
        enc = tokenizer(
            bdf["text"].tolist(),
            truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        feats = None
        if use_structured_feats:
            feats = torch.tensor(
                np.stack([bdf["age"].values, bdf["n_dx"].values], axis=1),
                dtype=torch.float32, device=device
            )
        out = model(**enc, feats=feats)
        logits_all.append(out["mort_logits"].detach().cpu().numpy())
    logits = np.concatenate(logits_all)
    prob = 1.0 / (1.0 + np.exp(-logits))
    return prob, logits

@torch.no_grad()
def predict_los_probs(model, tokenizer, df_part, max_len, batch_size=32, use_structured_feats=True):
    model.eval()
    logits_all = []
    for i in range(0, len(df_part), batch_size):
        bdf = df_part.iloc[i:i + batch_size]
        enc = tokenizer(
            bdf["text"].tolist(),
            truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        feats = None
        if use_structured_feats:
            feats = torch.tensor(
                np.stack([bdf["age"].values, bdf["n_dx"].values], axis=1),
                dtype=torch.float32, device=device
            )
        out = model(**enc, feats=feats)
        logits_all.append(out["los_logits"].detach().cpu().numpy())
    los_logits = np.concatenate(logits_all)
    expz = np.exp(los_logits - los_logits.max(axis=1, keepdims=True))
    los_prob = expz / expz.sum(axis=1, keepdims=True)
    los_pred = np.argmax(los_prob, axis=1)
    return los_prob, los_pred


# ------------------------------
# Main
# ------------------------------
def main():
    edges = cfg.los_bin_edges
    K = len(edges) - 1
    print("LOS bins:", edges, "K=", K, "labels:", cfg.los_bin_labels)

    df = pd.read_csv(cfg.csv_path)

    # admission-level
    df_adm = make_admission_level_df(df, max_dx=cfg.max_dx, edges=edges)
    print("admissions:", len(df_adm))
    print("mortality rate:", df_adm["y_mort"].mean())
    print("LOS class counts:", df_adm["y_los_cls"].value_counts().sort_index().to_dict())

    # split
    df_train, df_val, df_test = split_subject_wise_stratified_by_race(
        df_adm, seed=cfg.seed, test_size=cfg.test_size, val_size=cfg.val_size
    )
    print("train/val/test:", len(df_train), len(df_val), len(df_test))

    # optional tokenizer expansion
    if cfg.add_code_tokens_to_tokenizer:
        all_tok = []
        for t in df_train["text"].astype(str).tolist():
            for w in t.split():
                if w.startswith("DX_") or w.startswith("CCSR_"):
                    all_tok.append(w)
        vocab = [w for w, c in Counter(all_tok).most_common(cfg.max_added_tokens)]
        added = tokenizer.add_tokens(vocab, special_tokens=False)
        print("Added tokens:", added)

    feat_cols = ["age", "n_dx"] if cfg.use_structured_feats else []
    base_cols = ["text", "y_mort", "y_los_cls"]

    train_ds = Dataset.from_pandas(df_train[base_cols + feat_cols])
    val_ds = Dataset.from_pandas(df_val[base_cols + feat_cols])
    test_ds = Dataset.from_pandas(df_test[base_cols + feat_cols])

    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)
    test_ds = test_ds.map(tokenize_batch, batched=True)

    cols = ["input_ids", "attention_mask", "y_mort", "y_los_cls"] + feat_cols
    train_ds.set_format(type="torch", columns=cols)
    val_ds.set_format(type="torch", columns=cols)
    test_ds.set_format(type="torch", columns=cols)

    collator = DataCollatorMultiTask(tokenizer, use_structured_feats=cfg.use_structured_feats)

    # pos_weight for mortality
    pos = df_train["y_mort"].sum()
    neg = len(df_train) - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32)
    print("pos_weight:", pos_weight.item())

    model = MultiTaskLLM(
        cfg.model_name,
        num_los_classes=K,
        pos_weight=pos_weight,
        use_focal=cfg.use_focal,
        focal_alpha=cfg.focal_alpha,
        focal_gamma=cfg.focal_gamma,
        use_structured_feats=cfg.use_structured_feats
    ).to(device)

    if cfg.add_code_tokens_to_tokenizer:
        model.backbone.resize_token_embeddings(len(tokenizer))

    args = make_training_args(
        output_dir="llm_multitask_out",
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.train_bs,
        per_device_eval_batch_size=cfg.eval_bs,
        gradient_accumulation_steps=cfg.grad_accum,
        eval_strategy="epoch",
        save_strategy="epoch",
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,
        max_grad_norm=cfg.max_grad_norm,
        fp16=use_fp16,
        bf16=use_bf16,
        load_best_model_at_end=True,
        metric_for_best_model="mort_ap",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
    )

    trainer = MultiTaskTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        w_mort=cfg.w_mort,
        w_los=cfg.w_los,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stop_patience)]
    )

    print("\n--- Training ---")
    trainer.train()

    # Threshold tuning on VAL
    print("\n--- Threshold tuning on VAL (mortality) ---")
    y_val = df_val["y_mort"].values.astype(int)
    mort_prob_val, _ = predict_mort_probs(model, tokenizer, df_val, cfg.max_len, cfg.eval_bs, cfg.use_structured_feats)

    if cfg.threshold_mode.lower() == "f1":
        t_star, f1v = find_best_threshold_by_f1(y_val, mort_prob_val)
        print(f"Best VAL threshold (F1): {t_star:.4f} F1={f1v:.4f}")
    elif cfg.threshold_mode.lower() == "recall":
        res = threshold_for_target_recall(y_val, mort_prob_val, target_recall=cfg.target_recall)
        t_star = res[0] if res else find_best_threshold_by_f1(y_val, mort_prob_val)[0]
        print("VAL threshold (recall policy):", t_star)
    elif cfg.threshold_mode.lower() == "precision":
        res = threshold_for_target_precision(y_val, mort_prob_val, target_precision=cfg.target_precision)
        t_star = res[0] if res else find_best_threshold_by_f1(y_val, mort_prob_val)[0]
        print("VAL threshold (precision policy):", t_star)
    else:
        raise ValueError("threshold_mode must be f1 | recall | precision")

    # TEST: overall metrics + CIs
    print("\n--- TEST: F1 & AUROC with 95% bootstrap CI ---")
    y_test_m = df_test["y_mort"].values.astype(int)
    y_test_l = df_test["y_los_cls"].values.astype(int)

    mort_prob_test, _ = predict_mort_probs(model, tokenizer, df_test, cfg.max_len, cfg.eval_bs, cfg.use_structured_feats)
    mort_pred_test = (mort_prob_test >= t_star).astype(int)
    print("Mortality CM:\n", confusion_matrix(y_test_m, mort_pred_test))

    los_prob_test, los_pred_test = predict_los_probs(model, tokenizer, df_test, cfg.max_len, cfg.eval_bs, cfg.use_structured_feats)

    mort_f1_ci = bootstrap_ci(
        y_test_m, mort_pred_test,
        metric_fn=lambda yt, yp: f1_score(yt, yp, zero_division=0),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed
    )
    mort_auc_ci = bootstrap_ci(
        y_test_m, mort_prob_test,
        metric_fn=lambda yt, ps: safe_roc_auc(yt, ps),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed
    )
    los_f1_ci = bootstrap_ci(
        y_test_l, los_pred_test,
        metric_fn=lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed
    )
    los_auc_ci = bootstrap_ci(
        y_test_l, los_prob_test,
        metric_fn=lambda yt, prob: multiclass_auc_ovr_macro(yt, prob),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed
    )

    mf1p, mf1l, mf1h = pct3(mort_f1_ci)
    maucp, maucl, mauch = pct3(mort_auc_ci)
    lf1p, lf1l, lf1h = pct3(los_f1_ci)
    lau_cp, lau_cl, lau_ch = pct3(los_auc_ci)

    print("\n=== Performance Values (%, 95% CI) ===")
    print("Length-of-Stay Prediction (multiclass bins):")
    print("  Macro F1 (%):", fmt_ci(lf1p, lf1l, lf1h))
    print("  AUROC  (%):  ", fmt_ci(lau_cp, lau_cl, lau_ch))

    print("\nMortality Prediction:")
    print("  F1 (%):     ", fmt_ci(mf1p, mf1l, mf1h))
    print("  AUROC (%):  ", fmt_ci(maucp, maucl, mauch))

    # Subgroup performance by race
    print("\n--- TEST: Subgroup performance by race ---")
    race_test = df_test["race"].fillna("UNKNOWN").astype(str).values

    tbl = subgroup_metrics_table(
        df_test,
        mort_prob=mort_prob_test,
        mort_pred=mort_pred_test,
        los_prob=los_prob_test,
        los_pred=los_pred_test,
        n_boot=cfg.n_boot,
        alpha=cfg.ci_alpha,
        seed=cfg.seed,
        min_n_ci=cfg.min_group_n
    )
    pd.set_option("display.width", 220)
    pd.set_option("display.max_columns", 80)
    print(tbl.to_string(index=False))

    # Disparity measures (max-min gap)
    print("\n--- Disparity measures (max - min across race groups) ---")
    min_n_gap = max(20, cfg.min_group_n // 2)

    mort_auc_gap = bootstrap_gap_by_group(
        y_test_m, mort_prob_test, race_test,
        metric_fn=lambda yt, ps: safe_roc_auc(yt, ps),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    mort_ap_gap = bootstrap_gap_by_group(
        y_test_m, mort_prob_test, race_test,
        metric_fn=lambda yt, ps: safe_ap(yt, ps),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    mort_f1_gap = bootstrap_gap_by_group(
        y_test_m, mort_pred_test, race_test,
        metric_fn=lambda yt, yp: f1_score(yt, yp, zero_division=0),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    los_f1_gap = bootstrap_gap_by_group(
        y_test_l, los_pred_test, race_test,
        metric_fn=lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    los_auc_gap = bootstrap_gap_by_group(
        y_test_l, los_prob_test, race_test,
        metric_fn=lambda yt, prob: multiclass_auc_ovr_macro(yt, prob),
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    tpr_gap = bootstrap_gap_by_group(
        y_test_m, mort_pred_test, race_test,
        metric_fn=lambda yt, yp: tpr_fpr(yt, yp)[0],
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )
    fpr_gap = bootstrap_gap_by_group(
        y_test_m, mort_pred_test, race_test,
        metric_fn=lambda yt, yp: tpr_fpr(yt, yp)[1],
        n_boot=cfg.n_boot, alpha=cfg.ci_alpha, seed=cfg.seed, min_n=min_n_gap
    )

    print("Mortality AUROC gap:", fmt_ci(*mort_auc_gap, d=4))
    print("Mortality AP gap:   ", fmt_ci(*mort_ap_gap, d=4))
    print("Mortality F1 gap:   ", fmt_ci(*mort_f1_gap, d=4))
    print("LOS Macro-F1 gap:   ", fmt_ci(*los_f1_gap, d=4))
    print("LOS AUROC gap:      ", fmt_ci(*los_auc_gap, d=4))
    print("TPR gap (EqOpp):    ", fmt_ci(*tpr_gap, d=4))
    print("FPR gap:            ", fmt_ci(*fpr_gap, d=4))

    # Calibration by race (mortality + LOS)
    calib_dir = os.path.join(cfg.save_dir, "calibration_by_race")
    os.makedirs(calib_dir, exist_ok=True)

    _ = calibration_by_race_binary(
        y_true=y_test_m,
        y_prob=mort_prob_test,
        race=race_test,
        out_dir=calib_dir,
        n_bins=cfg.calib_bins,
        min_n=cfg.min_group_n,
        top_k_overlay=cfg.calib_topk_overlay,
        prefix="mortality"
    )

    _ = calibration_by_race_multiclass(
        y_true_cls=y_test_l,
        prob_2d=los_prob_test,
        race=race_test,
        out_dir=calib_dir,
        n_bins=cfg.calib_bins,
        min_n=cfg.min_group_n,
        top_k_overlay=cfg.calib_topk_overlay,
        prefix="los"
    )

    print(f"\nSaved calibration plots/metrics to: {calib_dir}")

    # Save model + tokenizer
    os.makedirs(cfg.save_dir, exist_ok=True)
    trainer.save_model(cfg.save_dir)
    tokenizer.save_pretrained(cfg.save_dir)
    print("\nSaved to:", cfg.save_dir)


if __name__ == "__main__":
    main()

device: cuda
bf16: True fp16: False
LOS bins: (0.0, 3.0, 7.0, 14.0, inf) K= 4 labels: ('0-3', '3-7', '7-14', '14+')
admissions: 11818
mortality rate: 0.023184972076493483
LOS class counts: {0: 6271, 1: 3372, 2: 1430, 3: 745}
train/val/test: 8318 1698 1802


Map:   0%|          | 0/8318 [00:00<?, ? examples/s]

Map:   0%|          | 0/1698 [00:00<?, ? examples/s]

Map:   0%|          | 0/1802 [00:00<?, ? examples/s]

pos_weight: 46.531429290771484


...

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



--- Training ---


Epoch,Training Loss,Validation Loss,Mort Auc,Mort Ap,Mort F1@0.5,Los Macro F1,Los Auc
1,1.012600,0.968992,0.880971,0.181549,0.000000,0.424786,0.826194
2,0.904800,0.913378,0.927646,0.248031,0.340741,0.421053,0.826991
3,0.897600,0.858570,0.970837,0.531372,0.539007,0.472655,0.832703
4,0.873700,0.860641,0.968119,0.583132,0.410714,0.491877,0.836265
5,0.834300,0.884295,0.972725,0.601575,0.530303,0.482799,0.833879
6,0.774800,0.903413,0.973459,0.586082,0.393013,0.491783,0.829556
7,0.683600,0.971268,0.968550,0.573517,0.521008,0.478673,0.834080
8,0.636500,1.094242,0.969569,0.573391,0.531469,0.505054,0.820136
9,0.545700,1.170169,0.968871,0.630096,0.531250,0.484419,0.814574
10,0.440900,1.305079,0.959084,0.605995,0.588235,0.478868,0.806365



--- Threshold tuning on VAL (mortality) ---
Best VAL threshold (F1): 0.3945 F1=0.6087

--- TEST: F1 & AUROC with 95% bootstrap CI ---
Mortality CM:
 [[1738   15]
 [  30   19]]

=== Performance Values (%, 95% CI) ===
Length-of-Stay Prediction (multiclass bins):
  Macro F1 (%): 52.42 (49.45, 55.33)
  AUROC  (%):   83.26 (81.91, 84.58)

Mortality Prediction:
  F1 (%):      45.78 (30.77, 58.23)
  AUROC (%):   87.16 (80.61, 93.08)

--- TEST: Subgroup performance by race ---
                                     race    n  mort_rate  mort_auc  mort_auc_lo  mort_auc_hi  mort_ap  mort_ap_lo  mort_ap_hi  mort_f1  mort_f1_lo  mort_f1_hi  mort_precision  mort_recall  mort_tpr  mort_fpr  los_macro_f1  los_f1_lo  los_f1_hi  los_auc  los_auc_lo  los_auc_hi
                                    WHITE 1152   0.027778  0.840583     0.751873     0.921302 0.347921    0.200860    0.562712 0.415094    0.239952    0.571429         0.52381      0.34375   0.34375  0.008929      0.522704   0.486199   0.557094 0.